**Import dependencies**

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
from pathlib import Path

**Read data**

In [ ]:
tau_ext_data = pd.read_csv("tau_log_test.csv")
tau_ext_deepc_data = pd.read_csv("tau_ext_test_deepc.csv")

# put 'label' as the row index
tau_ext_data = tau_ext_data.set_index("label")
tau_ext_deepc_data = tau_ext_deepc_data.set_index("label")

# keep only the time columns (they look like t0.0, t0.1, ..., etc.)
time_cols = [c for c in tau_ext_data.columns if c.startswith("t")]
time_cols_deepc = [c for c in tau_ext_deepc_data.columns if c.startswith("t")]

# numeric sort for rows like 'tau_cmd_0', 'tau_cmd_1', ...
def row_sort_key(s):
    m = re.search(r"_(\d+)$", s)
    return int(m.group(1)) if m else 10**9

# select and sort the tau_ext
tau_ext = sorted([r for r in tau_ext_data.index if r.startswith("tau_ext")], key=row_sort_key)
tau_ext_deepc = sorted([r for r in tau_ext_deepc_data.index if r.startswith("tau_ext")], key=row_sort_key)

tau_ext = tau_ext_data.loc[tau_ext, time_cols].to_numpy()
tau_ext_deepc = tau_ext_deepc_data.loc[tau_ext_deepc, time_cols_deepc].to_numpy()

# print(tau_ext.shape)
# print(tau_ext_deepc.shape)

(7, 19080)
(7, 19000)


**Plot tau_ext for each joint**

In [14]:
save_dir = Path.cwd()
n_rows = 7

# Compare only the overlapping portion.
T = min(tau_ext.shape[1], tau_ext_deepc.shape[1])

# time vector in seconds
time = np.arange(T) / 1000.0  # seconds

for i in range(n_rows):
    fig, ax = plt.subplots(figsize=(6, 3), dpi=150)

    ax.plot(time, tau_ext[i, :T], label="Joint_Impedance", color="blue", linewidth=1.2)
    ax.plot(time, tau_ext_deepc[i, :T], label="Kernel_DeePC", color="orange", linewidth=1.2)

    ax.set_title(f"Friction in joint {i+1}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Torque [Nm]")
    ax.legend()
    ax.grid(True, alpha=0.3)

    out_path = save_dir / f"friction_joint_{i+1}.pdf"
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)